# Supply Chain Shipment Analysis
This notebook implements a medallion architecture (Bronze → Silver → Gold) to process and analyze supply chain shipment data.

**Data Source:** `suppy_chain_shipment_data` table

**Objective:** Create clean, typed data layers and calculate key supply chain KPIs including lead times, freight costs, and shipment mode distribution.

## Medallion Architecture Overview
* **Bronze Layer:** Raw data ingestion with minimal transformation (column name standardization)
* **Silver Layer:** Cleansed and typed data with proper data types for numeric and date fields
* **Gold Layer:** Business-ready aggregations and KPIs for analytics and reporting

# Bronze Layer

The Bronze layer ingests raw shipment data and performs basic cleansing: standardizing column names by removing special characters and spaces, converting to lowercase, and replacing spaces with underscores.

## Data Quality Observations
The raw data contains 10,324 records. Note that weight and freight cost columns are stored as strings and contain mixed formats. Date columns also have inconsistent formats that will need standardization in the Silver layer.

In [0]:
spark.read.table("suppy_chain_shipment_data").show()

In [0]:
df = spark.read.table("suppy_chain_shipment_data")
df.count()

In [0]:
df.printSchema()

In the next steps, we need to rename the column names to delete or replace the spaces and special characters

In [0]:
df.columns

In [0]:
import re

# Clean column names: lowercase, replace spaces with _, remove special characters
clean_columns = [re.sub(r'[#/(),;{}]', '', col).strip().replace(' ', '_').lower() 
                 for col in df.columns]

# Apply new column names to the DataFrame
df_clean = df.toDF(*clean_columns)

# Verify the new column names
df_clean.printSchema()

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("bronze_shipments")

In [0]:
df_clean.count()

# Silver Layer

The Silver layer transforms bronze data into analytics-ready format by:
* Converting weight and freight cost from string to double (numeric)
* Standardizing date columns from multiple formats into proper date types
* Filtering out invalid values using `try_cast` and `try_to_date` to handle conversion errors gracefully

## Standardizing Date Formats
The source data contains dates in two formats:
* **M/d/yyyy** format: PQ first sent date, PO sent date
* **d-MMM-yy** format: Scheduled delivery, actual delivery, and recorded dates

Using `try_to_date` ensures graceful handling of invalid dates.

In [0]:
df_bronze = spark.read.table("bronze_shipments")

In [0]:
df_bronze.select("weight_kilograms").distinct().show()

In [0]:
df_bronze.select("freight_cost_usd").distinct().show()

We create a new df for the Silver Layer and then we change the data type of the weight, freight and date columns

In [0]:
df_silver = spark.read.table("bronze_shipments")

In [0]:
from pyspark.sql.functions import expr

df_silver = df_silver.withColumn("weight_kilograms", expr("try_cast(weight_kilograms as double)"))
df_silver = df_silver.withColumn("freight_cost_usd", expr("try_cast(freight_cost_usd as double)"))

In [0]:
df_silver.select("weight_kilograms", "freight_cost_usd").printSchema()

In [0]:
from pyspark.sql.functions import try_to_date

df_silver = df_silver.withColumn("pq_first_sent_to_client_date", try_to_date(df_silver["pq_first_sent_to_client_date"], "M/d/yyyy"))
df_silver = df_silver.withColumn("po_sent_to_vendor_date", try_to_date(df_silver["po_sent_to_vendor_date"], "M/d/yyyy"))
df_silver = df_silver.withColumn("scheduled_delivery_date", try_to_date(df_silver["scheduled_delivery_date"], "d-MMM-yy"))
df_silver = df_silver.withColumn("delivered_to_client_date", try_to_date(df_silver["delivered_to_client_date"], "d-MMM-yy"))
df_silver = df_silver.withColumn("delivery_recorded_date", try_to_date(df_silver["delivery_recorded_date"], "d-MMM-yy"))

In [0]:
df_silver.select("pq_first_sent_to_client_date", "po_sent_to_vendor_date", "scheduled_delivery_date", "delivered_to_client_date", "delivery_recorded_date").printSchema()



In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_shipments")

# Gold Layer

The Gold layer contains business KPIs and aggregations optimized for dashboards and reporting. Each KPI is calculated from the clean Silver layer data.

## First KPI in the Gold Layer is Lead time days

In [0]:
df_gold = spark.read.table("silver_shipments")

In [0]:
from pyspark.sql.functions import datediff, col

df_gold = df_gold.withColumn("lead_time_days",datediff(col("delivered_to_client_date"), col("po_sent_to_vendor_date")))

In [0]:
df_gold.select("po_sent_to_vendor_date", "delivered_to_client_date", "lead_time_days").show(10)


In [0]:
from pyspark.sql.functions import col

df_gold_filtered = df_gold.filter(col("lead_time_days").isNotNull())

df_gold_filtered.select("po_sent_to_vendor_date", "delivered_to_client_date", "lead_time_days").show(10)

In [0]:
from pyspark.sql.functions import count, when

df_gold.select(
    count(when(col("po_sent_to_vendor_date").isNotNull(), 1)).alias("po_date_count"),
    count(when(col("delivered_to_client_date").isNotNull(), 1)).alias("delivered_date_count"),
    count(when(col("lead_time_days").isNotNull(), 1)).alias("lead_time_count")
).show()

## Second KPI in the Gold Layer is Country & Product Group Aggregations
Aggregates total freight costs, line item values, and average lead times by country and product group for regional performance analysis.

In [0]:
from pyspark.sql.functions import sum, avg

df_country_product_group_kpi = df_gold.groupBy("country", "product_group") \
    .agg(
        sum("freight_cost_usd").alias("total_freight_cost"),
        sum("line_item_value").alias("total_line_item_value"),
        avg("lead_time_days").alias("avg_lead_time_days")
    )

df_country_product_group_kpi.show(10)

## Third KPI in the Gold Layer is Freight Cos Per Unit per product

This metric calculates the freight cost efficiency per unit shipped, helping identify products or shipments with disproportionately high shipping costs relative to quantity.

In [0]:
from pyspark.sql.functions import when, lit, col

df_gold = df_gold.withColumn("freight_cost_per_unit",when(col("line_item_quantity") > 0, lit(col("freight_cost_usd")/col("line_item_quantity"))).otherwise(lit(None)))

df_gold.select("item_description", "freight_cost_usd", "line_item_quantity", "freight_cost_per_unit").show(10)


## Fourth KPI in the Gold Layer is Shipment Mode Mix per Country

In [0]:
from pyspark.sql.functions import count

df_mode_count = df_gold.groupBy("shipment_mode","country").agg(count("*").alias("count"))

df_mode_count.show(10)

In [0]:
from pyspark.sql.functions import count

df_country_total = df_gold.groupBy("country").agg(count("*").alias("Total_shipments"))

df_country_total.show(10)

In [0]:
df_mode_mix = df_mode_count.join(df_country_total, on="country", how="left")
df_mode_mix = df_mode_mix.withColumn("pct_shipments", (col("count") / col("Total_shipments")) * 100)

df_mode_mix.show(10)

In [0]:
df_country_product_group_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_shipment_kpis")

df_mode_mix.write.format("delta").mode("overwrite").saveAsTable("gold_mode_mix")